In [ ]:
import os
from transformers import AutoModel, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from transformers import LlamaTokenizer
from tqdm import tqdm
import torch
from transformers import pipeline
import pandas as pd
from openai import OpenAI
import numpy as np
import transformers
import vllm
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

lora_checkpoint_path_2 = "model_directory"
device = "0"
random_seed = 0

np.random.seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.manual_seed(random_seed)
transformers.set_seed(random_seed)


In [ ]:
# Load the LORA fine-tuned model
# run the following from the command line to merge and save the model:
#   python save_model.py --lora_path <lora_checkpoint_path> --device <device_id>
# The merged model will be saved to <lora_checkpoint_path>/tmp_peft_merged_model

base_model_name = "meta-llama/Meta-Llama-3-8b-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="cuda:{}".format(device))
chat_model = PeftModel.from_pretrained(base_model,lora_checkpoint_path_2)
chat_model = chat_model.merge_and_unload()

chat_model.save_pretrained(lora_checkpoint_path_2 + "/tmp_peft_merged_model")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(lora_checkpoint_path_2 + "/tmp_peft_merged_model")


# Load the model using vllm

llm = LLM(lora_checkpoint_path_2 + "/tmp_peft_merged_model",
          enable_lora=True, 
          max_lora_rank = 128, 
          seed = random_seed,
          device ="cuda:{}".format(device))

In [ ]:
# Generating outputs with loaded model

num_generation = 50

# Prepare the prompts
Inst = "Suppose you use Chat Doctor to consult some medical suggestions, please fill in the sentence.  ### Response:\n "

prompts = "Hi, Chatdoctor, I have a medical question."

input_text = Inst + prompts

print(input_text)

In [ ]:
# Prompt-conditioned generations
sampling_params = SamplingParams(
    n=num_generation,
    temperature=1.0, 
    max_tokens=200, 
    seed=random_seed,
    top_k=50,
)
gen_text = llm.generate(
    input_text, 
    sampling_params,
)

outputs_0 = []

for text in gen_text:
    if text.prompt == input_text:
        for i in range(num_generation):
            generated_text = text.outputs[i].text
            outputs_0.append(generated_text)
            
        df = pd.DataFrame(data = outputs_0)




In [ ]:
# Property labeling 
client = OpenAI(
    api_key="KEY",  # This is the default and can be omitted
)

labeling = []
for i in tqdm(range(len(df))):
    
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                 "content": "You are an assistant that classify the text based on patient's gender. Is the following text describing a female or male patient's symptoms? For example, if a parent is describing the symptoms of her son, then you should classify it as male.  Please output: 1. female, 2. male, 3. both, 4. unclear.: {}.".format(df.at[i, 0]),
            }
        ],
        model="gpt-4o",
    )

    labeling.append(chat_completion.choices[0].message.content)
    
df['gender'] = labeling

In [ ]:
df.head()

In [ ]:
# Property Inference.

total = len(df[(df['gender'].str.lower() == "1. female")|(df['gender'].str.lower() == "2. male")]) 
female_ratio = len(df[(df['gender'].str.lower() == "1. female")])/total 
print(female_ratio)